# ETL (Extract, Transform, Load) Data Transactions (**Headers**) and Transactions Items (**Details**) iSeller Mitra
>An ETL proccess for get raw data transactions details (Transactions per Items) from API Endpoint `https://janjijiwamitra.isellershop.com/api/v2/GetOrders`

Updated at : 2026-10-16

Updated by : lahiardhan@jiwagroup.com

LATEST UPDATE :
> - 2026-10-13 : Fix Accuracy Details vs Headers (Discount/Promo Fan-out, Refund, Combo Allocation, Tax from API)
> - 2026-10-13 : Enhance API Fetching Completeness Check (Retry on API Error, Re-fetch Empty Response, Compare vs Data Lake)
> - 2026-10-16 : Fix Custom Discount (Kasir) Not Deducted on Details -> Use discounts[] Amount
> - 2026-10-16 : Fix Missing Orders on Window Boundary (xx:59:59) -> Window End at xx:59:59.999
> - 2026-10-16 : Simplify API Fetching Cell (Keep Retry, Re-fetch Empty & Per-Outlet Progress)


# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np

import pymysql

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import paramiko

import requests
import json

# CONFIG DB

In [ ]:
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../../db_config/config_db_datawarehouse.py")

# FUNCTION

In [ ]:
def convert_data_to_dataframe(data):
    df_all = pd.DataFrame(data)
    for column in ['transactions']:
        df_all[column] = df_all[column].apply(lambda x: x[0] if x else None)  
        if df_all[column][0] is not None: 
            if isinstance(df_all[column][0], dict): 
                for key in df_all[column][0].keys():
                    new_column = f"{column}_{key}"
                    df_all[new_column] = df_all[column].apply(lambda x: x[key] if x else None)
    return df_all

In [ ]:
def expand_nested_lists(df, column_names):
    expanded_data = []
    for index, row in df.iterrows():
        for column_name in column_names:
            data_list = row[column_name]
            if isinstance(data_list, list): 
                for item in data_list:
                    if item is not None:  
                        new_row = row.copy() 
                        for key, value in item.items():
                            new_column_name = f"{column_name}_{key}"  
                            new_row[new_column_name] = value
                        expanded_data.append(new_row)
            else:
                expanded_data.append(row)
    df_expanded = pd.DataFrame(expanded_data)
    return df_expanded

In [ ]:
def adjust_timezone(date_string, hours_offset):
    datetime_obj = datetime.datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S")
    adjusted_datetime = datetime_obj + datetime.timedelta(hours=hours_offset)
    return adjusted_datetime.strftime("%Y-%m-%d %H:%M:%S")

# MAIN PROGRESS

## 1. Get Auth Token & Date

In [ ]:
import os, getpass

# Token API iSeller Mitra: jangan ditulis di notebook. Set env var `access_token_mitra`
# (sama dengan Airflow Variable access_token_mitra), atau isi lewat prompt yang muncul.
access_token = os.getenv('access_token_mitra') or getpass.getpass('access_token_mitra: ')

In [ ]:
today = datetime.date.today() - datetime.timedelta(days=1)
today_string = today.strftime("%Y-%m-%d")
start_date = today
end_date = today

num_days = (end_date - start_date).days + 1

timezone_offset_hours = -7

start_date_list = []
end_date_list = []

#looping x days
for day_offset in range(num_days):
    for i in range(6): #LOOPING 6/DAYS/OUTLET 
        start_hour = i * 4
        end_hour = (i + 1) * 4 - 1

        start_date_string = (start_date + datetime.timedelta(days=day_offset)).strftime("%Y-%m-%d") + f" {start_hour:02}:00:00"
        adjusted_start_date = adjust_timezone(start_date_string, timezone_offset_hours)
        start_date_list.append(adjusted_start_date)

        end_date_string = (start_date + datetime.timedelta(days=day_offset)).strftime("%Y-%m-%d") + f" {end_hour:02}:59:59"
        adjusted_end_date = adjust_timezone(end_date_string, timezone_offset_hours)
        end_date_list.append(adjusted_end_date + ".999")


print(f"today: {today_string}")
print(f"start date list: {start_date_list}")
print(f"end date list : {end_date_list}")


## 2. Get Outlet Master Data

In [ ]:
url = 'https://janjijiwamitra.isellershop.com/api/v2/getstoreinfo'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

process = requests.get(url, headers=headers)
string_to_json = json.loads(process.text)
df = pd.DataFrame(list(string_to_json.items()),columns = ['name','value'])
x = df['value'][0]
df_x = pd.DataFrame(list(x.items()),columns = ['name','value'])
outlet_data = df_x['value'][0]

In [ ]:
df_outlet = pd.DataFrame(outlet_data)
df_outlet['outlet_code'] = df_outlet['outlet_code'].str.zfill(5)
df_outlet = df_outlet[df_outlet['outlet_code'] != 'Main Outlet' ]
active_outlet_list = df_outlet['outlet_id'].unique()
print(f"total outlet : {len(active_outlet_list)}")

## 3. Get Items Master Data

In [ ]:
url = 'https://janjijiwamitra.isellershop.com/api/v2/GetProducts'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

flag = 0
df_product_data = pd.DataFrame() 
get_product = []

for i in range(1, 5): #karena saat ini total products dan bundling kurang lebih hanya 1200 maka buat looping 4

    print(flag)

    data = {
    'modified_after': '2023-01-01',
    'track_inventory': 'FALSE',
    'page_size' : 500, #max pages 
    'page' : i
    }

    process = requests.post(url, headers=headers, data=data)
    string_to_json = json.loads(process.text)
    df = pd.DataFrame(list(string_to_json.items()), columns=['name', 'value']) 
    try:
        product_data = pd.DataFrame(df['value'][0])
        df_product_data = pd.concat([product_data, df_product_data], ignore_index=False) 
        get_product.append(df['value'][0])
    except:
        pass

    flag += 1

print(f"Total Master Data : {len(df_product_data)}")

In [ ]:
#membagi master data products (per-sku) dan bundling (per bundling_code)
 
df_bundlings = df_product_data[df_product_data['type'] == 'comboset'] #data bundling
df_products = df_product_data[df_product_data['type'] != 'comboset'] #data products 

#normalize master data products
df_products = df_products[
    [
        'product_id', 'product_type', 'sku', 'name', 'type', 'taxable', 'price', 'sold_count', 'modified_date'
    ]
]
df_products_anomaly = df_products[(df_products['sku'] == '') | (df_products['price'] == 0)]
df_products =  df_products[df_products['sku'] != '']
df_products = df_products[df_products['price'] != 0]

#normalize master data bundlings
df_bundlings = df_bundlings[['product_id', 'name', 'sku', 'taxable', 'sold_count', 'bundles', 'modified_date']]
df_bundlings.columns = ['bundling_id', 'bundling_name', 'bundling_code', 'taxable', 'sold_count', 'bundles', 'modified_date']
columns_to_expand = ['bundles']
df_bundling_exp = expand_nested_lists(df_bundlings, columns_to_expand)
x_columns = [col for col in df_bundling_exp.columns if col.startswith('bundles_')]
for col in x_columns:
    new_col_name = col.replace('bundles_', '')
    df_bundling_exp.rename(columns={col: new_col_name}, inplace=True)
df_bundling_exp.drop('bundles', axis=1, inplace=True)
df_bundling_final = pd.merge(df_bundling_exp, df_products[['product_id', 'price']], on='product_id', how='left')
df_bundlings = df_bundling_final[
    [
        'bundling_id', 'bundling_name', 'bundling_code', 'taxable', 'product_id', 'sku', 
        'category', 'product_name', 'variant_name', 'quantity', 'price', 'taxable', 
        'sold_count', 'modified_date'
    ]
]
df_bundlings['bundling_code'] = df_bundlings.apply(lambda row: row['bundling_code'][:14] if pd.notnull(row['bundling_code']) else None, axis=1)

print(f"Total Master Data Product : {len(df_products)}")
print(f"Total Master Data Bundling : {df_bundlings['bundling_id'].nunique()}")
print(f"Total Product dengan SKU Null : {len(df_products[df_products['sku'].isna()])}")

## 4. Get Order Data

### 4.1 Get All Order Data

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import time

url = 'https://janjijiwamitra.isellershop.com/api/v2/GetOrders'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

# 1 task = 1 outlet x 1 jendela waktu
tasks = [
    (outlet_id, start_date_list[i], end_date_list[i])
    for i in range(len(start_date_list))
    for outlet_id in active_outlet_list
]
print(f"Total API calls: {len(tasks)}")


def fetch_orders(task):
    outlet_id, start_dt, end_dt = task
    data = {
        'created_after': start_dt,
        'created_before': end_dt,
        'timezone': '7',
        'includes': 'orderdetails, promotiondetails, transactions, discountdetails',
        'outlet_id': outlet_id
    }
    for attempt in range(3):
        try:
            result = requests.post(url, headers=headers, data=data, timeout=30).json()
            # API bisa balas 200 tapi isinya error (rate limit, token, dll.) -> anggap gagal, coba lagi
            if result.get('status') is False or result.get('error_message'):
                raise ValueError(result.get('error_message'))
            return list(result.values())[1]
        except Exception as e:
            if attempt == 2:
                raise Exception(f"GAGAL outlet={outlet_id} {start_dt}~{end_dt}: {e}")
            time.sleep(2 ** (attempt + 1))


# tarik semua, dengan progress per outlet
from concurrent.futures import as_completed

get_data = [None] * len(tasks)
calls_per_outlet = len(start_date_list)
outlet_done = {outlet_id: 0 for outlet_id in active_outlet_list}
n_outlet_selesai = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_orders, task): i for i, task in enumerate(tasks)}
    for future in tqdm(as_completed(futures), total=len(tasks), desc="Fetching"):
        i = futures[future]
        get_data[i] = future.result()

        outlet_id = tasks[i][0]
        outlet_done[outlet_id] += 1
        if outlet_done[outlet_id] == calls_per_outlet:
            n_outlet_selesai += 1
            print(f"  [outlet done] {outlet_id} ({n_outlet_selesai}/{len(active_outlet_list)} outlets)")

# panggilan yang hasilnya 0 order diulang sekali (bisa jadi tadi gagal diam-diam)
empty_idx = [i for i, orders in enumerate(get_data) if not orders]
print(f"Panggilan dengan 0 order: {len(empty_idx)} dari {len(tasks)} -> diulang sekali")

recovered = 0
with ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(fetch_orders, [tasks[i] for i in empty_idx])
    for i, orders in zip(empty_idx, results):
        if orders:
            get_data[i] = orders
            recovered += 1

print(f"Ternyata ada order di {recovered} panggilan (sudah dipakai). Sisa 0 order: {sum(1 for o in get_data if not o)}")


In [ ]:
data = get_data.copy()

In [ ]:
#code untuk menyimpan dan load data hasil get saat testing
#menyimpan get data menjadi file (untuk local)
# with open('data_14-18.json', 'w') as file:
#     json.dump(data, file, indent=4)

#load data yg telah disimpan 
# with open('data.json', 'r') as file:
#     data = json.load(file)

### 4.2 Normalize Order Data

In [ ]:
df_headers = pd.DataFrame()
df_details = pd.DataFrame()
df_product_combos = pd.DataFrame()
df_promo = pd.DataFrame()
df_discount = pd.DataFrame()

try:
    
    for headers in data:
        if headers:
            headers_list_cols = [
                'outlet_id', 'order_reference','outlet_code', 'outlet_name', 'order_date', 'closed_date', 'status', 'channel', 'channel_detail', 'payment_status', 'fulfillment_status',
                'notes', 'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number', 'customer_email', 'cashier_id',
                'total_order_amount', 'total_discount_amount', 'total_promotion_amount', 'subtotal', 'total_tax_amount', 'total_amount'
            ]
            # Data headers dimasukkan ke dalam DataFrame df_headers
            df_headers = pd.concat([df_headers, pd.json_normalize(headers, record_path=['transactions'], meta=headers_list_cols , meta_prefix= 'transactions_')])

            # Data order_details dimasukkan ke dalam DataFrame df_details
            df_details = pd.concat([df_details, pd.json_normalize(headers, record_path=['order_details'])])

            # # Loop melalui setiap detail di headers
            for details in headers:
                if len(details['order_details']) > 0: 
                    # Data product_combos dimasukkan ke dalam DataFrame df_product_combos
                    df_product_combos = pd.concat([df_product_combos, pd.json_normalize(details['order_details'], record_path='product_combos', meta='order_detail_id')])
                    # Data promotions dimasukkan ke dalam DataFrame df_promo
                    df_promo = pd.concat([df_promo, pd.json_normalize(details['order_details'], record_path='promotions', meta='order_detail_id')])
                    # Data discounts dimasukkan ke dalam DataFrame df_discount
                    df_discount = pd.concat([df_discount, pd.json_normalize(details['order_details'], record_path='discounts', meta='order_detail_id')])

# except Exception as e:
#     print(f"Error occurred: {e}")

except Exception as e:
    import traceback; traceback.print_exc()
    print("elemen yang bikin error:", type(headers).__name__, str(headers)[:300])

### 4.3 Normalize Headers Data

In [ ]:
# get data headers
df_headers = df_headers[['order_id', 'transaction_id', 'transaction_date', 'transactions_order_reference',
            'transactions_outlet_id', 'transactions_outlet_code', 'transactions_outlet_name',
            'transactions_channel', 'transactions_channel_detail', 'transactions_order_date',
            'transactions_closed_date', 'transactions_status', 'transactions_payment_status', 
            'transactions_fulfillment_status', 'type', 'status',  'gateway', 'payment_type_name', 
            'transactions_notes', 'transactions_customer_id', 'transactions_customer_first_name', 
            'transactions_customer_last_name', 'transactions_customer_phone_number', 
            'transactions_customer_email', 'transactions_cashier_id', 'transactions_total_order_amount',
            'transactions_total_discount_amount', 'transactions_total_promotion_amount', 'transactions_subtotal',
            'transactions_total_tax_amount', 'transactions_total_amount', 'mdr']] 

df_headers.columns = [
                     'order_id','transaction_id', 'transaction_date', 'order_reference', 'outlet_id', 'outlet_code',
                     'outlet_name', 'channel', 'channel_detail','order_date', 'closed_date', 'status', 'payment_status', 'fulfillment_status', 
                     'transactions_type', 'transactions_status', 'transactions_gateway', 'transactions_payment_type_name', 
                     'notes', 'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number','customer_email', 'cashier_id', 
                     'total_order_amount', 'total_discount_amount', 'total_promotion_amount', 'subtotal', 'total_tax_amount', 
                     'total_amount', 'mdr'
            ]

df_headers['order_date'] = pd.to_datetime(df_headers['order_date'])
df_headers['closed_date'] = pd.to_datetime(df_headers['closed_date'])
df_headers['outlet_code'] = df_headers['outlet_code'].str.zfill(5)

gmt_plus_7 = timezone('Etc/GMT-7')

df_headers['transaction_date'] = pd.to_datetime(df_headers['transaction_date'])
df_headers['transaction_date'] = df_headers['transaction_date'].dt.tz_localize('UTC').dt.tz_convert(gmt_plus_7)

df_headers['customer_first_name'] = df_headers['customer_first_name'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))
df_headers['customer_last_name'] = df_headers['customer_last_name'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))
df_headers['notes'] = df_headers['notes'].astype(str).apply(lambda x: x.encode('ascii', 'ignore').decode('ascii'))

In [ ]:
#set sales type on each order
# df_headers_sjw = df_details[['order_id', 'product_id']]

product_id_sjw_sjo = {
    '1d2664fc-f2c4-4d63-93b0-0150c5bad9ca',
    '514f7f40-fcf1-4b24-a881-0e32a6e2da88',
    'b7a5bd88-cda9-4623-814a-734767026910',
    '134f7a5b-09a2-455c-8f70-93a01468131e',
    'b26ce832-29a4-46f7-be9f-e9f4d72c1dca',
    '64b0cc8b-88e4-498a-b5b5-ff093b5915e5',
    '2306edf4-9f70-4c79-88d9-2a836a292816',  # added at 2024-03-13
    '9eef163d-fd2f-4e52-98e6-4593679a16b5',  # added at 2024-04-01 semua SJO
    '9ea8be5c-8bc1-4f14-b564-5eb75a19d907',
    'a96b7a4e-b86d-4171-956b-77a09d1c54f9',
    '6b07599f-14f8-44f9-ab5b-4389bf7e82d1',
    '328e5672-48a1-4683-b65d-c4a45f344ca1',  # added at 2024-04-02
    'baa1d6c2-7b48-40af-a6f3-b9e481f93e61',
    '3ab4aa05-f664-4eaf-ad7e-bdf643d421d4',
    '2fcb7dad-bd59-44c9-a846-1279f9840d37',
    'c038d0fa-a12a-4644-9bbb-3878b3064912',
    '4521dc51-b3d6-4493-814a-617fde292c04',  # added at 2024-05-14
    '62701d95-4d24-449e-b0db-ea8994d00b62',  # added at 2024-07-08
    '8f5fe78b-7d7c-46ed-9f5b-c9f8d195b60d',  # added at 2024-08-04
    '44d2e817-8745-4cbf-aa36-d4abfc75aebf',
    'a26111f7-3aa5-48b0-a77b-2f952aab5eae',
    '74ee1eae-21b8-467a-828c-b59cf780d065',  # added at 2024-08-16
    'a89893bf-9026-4a5c-9f0c-949c0f4539d7',
    '150c96b0-fe81-4835-b3cc-593bfb6c45db',  # added at 2024-09-14
    '999251a7-5e8b-4267-b67c-b33cedead904',  # added at 2024-10-03
    '0c8a73fe-ae1b-4287-a6af-7d48fce86de5',
    'c9d038f6-fea1-494f-813c-ad0aed0dde1e',  # added at 2024-10-23
    'a94730e0-4f5b-4af9-b70b-68dbea299b78',
    '5842673c-8a8b-423b-9804-1c0d484c60ac',  # added at 2024-11-18
    'bae1b783-e862-4c5f-bb46-b20bb7e02b63',
    '00295e2b-66bb-489f-9615-6e25676c8331',
    'd8a7f659-c1dd-448a-9110-688fd1105bf8',
    '47855957-b056-49bc-8be3-c8ea68e0160b',
    '5864c6dc-216e-4764-9c8b-71bc9d77c41a',  # added at 2025-02-26
    '3ffe31ac-f4c6-4206-959f-26f2ec6224fc',
    '64b5deb0-fcef-412f-bd2e-714a9b65a6b9',  # added at 2025-06-23
    '7ddc0da7-72bf-4518-ac37-c1c62e48d418',  # added at 2026-08-20 (Basreng Pedas Daun Jeruk, ITM010872)
    '7fed141a-8b22-4cee-80c0-df4242e136ec',  # added at 2026-10-09 (Es Kopi Susu Honey, FCH040025)
    '5faac2db-7827-4414-a5ac-1345a79299dd',  # added at 2026-10-09 (Matcha - KSJ, FCH040036)
    '06aa6648-b40a-4d02-b9b7-3105fa83c762'   # added at 2026-10-09 (Matcha - SJO, ITM022191)
}

# is_sjw = 1 nilai per order: order berisi minimal 1 produk SJW -> Sejuta Jiwa
df_headers_sjw = (df_details['product_id'].isin(product_id_sjw_sjo)
                  .groupby(df_details['order_id']).any()
                  .rename('is_sjw').reset_index())
df_merged = pd.merge(df_headers, df_headers_sjw, on='order_id', how='left')
df_merged[df_merged['is_sjw'] == True]['transactions_payment_type_name'].unique()

#create conditions for set sales type (Per 5 Oct 2023)
conditions = [
    (df_merged['transactions_payment_type_name'] == 'Go Food'),
    (df_merged['transactions_payment_type_name'] == 'Grab Food'),
    (df_merged['transactions_payment_type_name'] == 'Shopee Food'),
    (df_merged['transactions_payment_type_name'] == 'Jiwa+'),
    (df_merged['transactions_payment_type_name'] == 'Grab Dine Out'),
    (df_merged['transactions_payment_type_name'] == 'Tokopedia Go'),
    (df_merged['transactions_payment_type_name'] == 'QPON')
]

values = ['Go Food', 'Grab Food', 'Shopee Food', 'Jiwa+', 'Grab Dine Out', 'Tokopedia Go', 'Qpon']

df_merged['sales_type'] = np.select(conditions, values, default='Walk In')

df_merged.loc[df_merged['is_sjw'] == True, 'sales_type'] = 'Sejuta Jiwa'

In [ ]:
#finalize headers
df_final_headers = df_merged[
    [
       'order_id', 'transaction_id', 'transaction_date',
       'order_reference', 'outlet_id', 'outlet_code', 'outlet_name', 'channel',
       'channel_detail', 'order_date', 'closed_date', 'status',
       'payment_status', 'fulfillment_status', 'transactions_type',
       'transactions_status', 'transactions_gateway', 'sales_type',
       'transactions_payment_type_name', 'notes', 'customer_id',
       'customer_first_name', 'customer_last_name', 'customer_phone_number',
       'customer_email', 'cashier_id', 'total_order_amount',
       'total_discount_amount', 'total_promotion_amount', 'subtotal',
       'total_tax_amount', 'total_amount', 'mdr'
    ]
]
df_final_headers['load_data_at'] = datetime.datetime.now()

df_final_headers.drop_duplicates(inplace=True)

### 4.4 Normalize Details Data 

In [ ]:
# Dibutuhkan proses grouping karena Mitra masih menggabungkan discount dalam promotion fields sehingga 1 order_details memiliki multiple rows

def _join_text(sep):
    # gabung nilai teks unik (urutan asli), abaikan kosong
    return lambda s: sep.join(dict.fromkeys(str(v) for v in s if pd.notna(v) and str(v) != ''))

# 1 row per order_detail_id: amount di-SUM, teks digabung -> merge ke details tidak fan-out
if len(df_promo):
    df_grouping_promo = df_promo.groupby('order_detail_id', sort=False).agg(
        promotion_id=('promotion_id', _join_text(', ')),
        promotion_name=('promotion_name', _join_text('-')),
        promotion_type=('promotion_type', _join_text(', ')),
        promotion_amount=('promotion_amount', 'sum'),
    ).reset_index()
else:
    df_grouping_promo = pd.DataFrame(columns=['order_detail_id', 'promotion_id', 'promotion_name', 'promotion_type', 'promotion_amount'])

# discount juga di-group: sebelumnya di-merge mentah, detail dengan > 1 discount jadi > 1 row
if len(df_discount):
    df_grouping_discount = df_discount.groupby('order_detail_id', sort=False).agg(
        discount_name=('discount_name', _join_text(', ')),
        discount_amount=('discount_amount', 'sum'),
    ).reset_index()
else:
    df_grouping_discount = pd.DataFrame(columns=['order_detail_id', 'discount_name', 'discount_amount'])

print(f"promo   : {len(df_promo):,} row -> {len(df_grouping_promo):,} order_detail")
print(f"discount: {len(df_discount):,} row -> {len(df_grouping_discount):,} order_detail")

In [ ]:
#merge promo and discount to order details
df_merge_details_promo = df_details.merge(df_grouping_promo, on='order_detail_id', how='left')
df_merge_details_promo_and_disc = df_merge_details_promo.merge(df_grouping_discount, on='order_detail_id', how='left')

df_merge_details_promo_and_disc = df_merge_details_promo_and_disc[['order_id', 'order_detail_id', 'product_id', 'product_name',
       'product_generic_name', 'product_type', 'product_variant_id',
       'product_variant_name', 'product_modifier_id', 'product_modifier_name',
       'notes', 'sku', 'fulfillment_status', 'quantity',
       'base_price', 'total_order_amount', 'subtotal', 'additional_charge',
       'discount_percentage', 'discount_order_amount',
       'taxable', 'tax_percentage', 'tax_amount','type', 'promotion_id', 'promotion_name',
       'promotion_type', 'promotion_amount', 'discount_name',
       'discount_amount_y', 'modifier_product_sku']]

print(f"details: {len(df_details):,} row -> setelah merge promo & discount: {len(df_merge_details_promo_and_disc):,} row (harus sama)")

In [ ]:
#get and merge headers attribute 
right_cols = ['order_id', 'transaction_date', 'order_reference',
       'outlet_id', 'outlet_code', 'outlet_name', 'channel', 'channel_detail',
       'order_date', 'closed_date', 'status', 'payment_status',
       'fulfillment_status', 'transactions_type', 'transactions_status', 'sales_type', 'transactions_payment_type_name',
       'notes', 'customer_id', 'customer_first_name', 'customer_last_name',
       'customer_phone_number', 'customer_email', 'cashier_id']

df_details_2 = df_merge_details_promo_and_disc.merge(df_final_headers[right_cols], on = 'order_id', how = 'left')

In [ ]:
#normalize combos 
df_merge_details_and_combos = df_details_2.merge(df_product_combos, on= 'order_detail_id', how='left')

df_merge_details_and_combos['bundling_id'] = ''
for index, row in df_merge_details_and_combos.iterrows():
    if row['type'] == 'comboset':
        df_merge_details_and_combos.at[index, 'bundling_id'] = df_merge_details_and_combos.at[index, 'product_id_x']
        df_merge_details_and_combos.at[index, 'product_id_x'] = df_merge_details_and_combos.at[index, 'product_id_y']
        df_merge_details_and_combos.at[index, 'product_name_x'] = df_merge_details_and_combos.at[index, 'product_name_y']
        df_merge_details_and_combos.at[index, 'quantity_x'] = df_merge_details_and_combos.at[index, 'quantity_y']
        df_merge_details_and_combos.at[index, 'product_variant_name'] = df_merge_details_and_combos.at[index, 'variant_name']
        df_merge_details_and_combos.at[index, 'sku'] = df_merge_details_and_combos.at[index, 'product_sku']

In [ ]:
#menghapus row jika detail product hanya berisi variant non-sku (['Normal', 'Less Ice', 'Less Sugar', '100%', '25%', '50%']) agar tidak duplicate 
df_filtered = df_merge_details_and_combos[~((df_merge_details_and_combos['product_modifier_id'] != '') & (df_merge_details_and_combos['base_price'] == 0))]

df_filtered = df_filtered[['order_id', 'transaction_date', 'order_reference', 'outlet_id',
       'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
       'closed_date', 'status', 'payment_status', 'transactions_type', 
       'transactions_status', 'sales_type', 'transactions_payment_type_name',
       'customer_id', 'customer_first_name', 'customer_last_name', 'customer_phone_number',
       'customer_email', 'cashier_id', 'order_detail_id', 'product_id_x', 'bundling_id','product_name_x',
       'product_generic_name','type', 'product_type', 'product_variant_id',
       'product_variant_name', 'product_modifier_id', 'product_modifier_name',
       'notes_x', 'sku', 'fulfillment_status_x', 'quantity_x', 'base_price',
       'total_order_amount','discount_name', 'discount_percentage', 'discount_order_amount',
       'discount_amount_y', 'promotion_id', 'promotion_name', 'promotion_type', 
       'promotion_amount', 'subtotal', 'taxable', 'tax_percentage', 
       'tax_amount', 'additional_charge', 'modifier_product_sku']]

df_filtered.columns = ['order_id', 'transaction_date', 'order_reference', 'outlet_id',
       'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
       'closed_date', 'status', 'payment_status', 'transactions_type',
       'transactions_status', 'sales_type', 'transactions_payment_type_name',
       'customer_id', 'customer_first_name', 'customer_last_name',
       'customer_phone_number', 'customer_email', 'cashier_id',
       'order_detail_id', 'product_id', 'bundling_id', 'product_name',
       'product_generic_name', 'type', 'product_type', 'product_variant_id',
       'product_variant_name', 'product_modifier_id', 'product_modifier_name',
       'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
       'total_order_amount', 'discount_name', 'discount_percentage', 'discount_order_amount',
       'discount_amount', 'promotion_id', 'promotion_name', 'promotion_type',
       'promotion_amount', 'subtotal', 'taxable', 'tax_percentage',
       'tax_amount', 'additional_charge', 'modifier_product_sku']

In [ ]:
# join base price products from Master Products  
df_filtered = pd.merge(df_filtered, df_products[['product_id', 'price']], on = 'product_id', how='left')

In [ ]:
# Normalize calculation
numeric_columns = df_filtered.select_dtypes(include='number').columns
string_columns = df_filtered.select_dtypes(include='object').columns

df_filtered[numeric_columns] = df_filtered[numeric_columns].fillna(0)
df_filtered[string_columns] = df_filtered[string_columns].fillna('')

# Tarif pajak dari API: `tax_percentage` per order_detail (untuk combo = tarif level paket).
# -- versi lama (tarif dari list produk 11%, list-nya ada di cell sebelumnya sebagai komentar):
# df_filtered['divisor'] = df_filtered['product_id'].apply(lambda x: 1.1 if x not in product_id_sjw_sjo_v2 else 1.11)
df_filtered['tax_rate'] = pd.to_numeric(df_filtered['tax_percentage'], errors='coerce').fillna(0) / 100
df_filtered['divisor'] = 1 + df_filtered['tax_rate']
print("tarif pajak dari API (tax_percentage -> jumlah row):", df_filtered['tax_percentage'].value_counts().to_dict())

# Combo: nilai paket dari API (total_order_amount level detail, sebelum pajak) dibagi ke komponen
# proporsional harga master x qty. Harga master hanya bobot -> total komponen = harga paket = header.
# Kalau ada komponen tanpa harga master, paket dibagi rata.
is_combo = df_filtered['type'] == 'comboset'
df_filtered['combo_share'] = 1.0
if is_combo.any():
    combo = df_filtered[is_combo]
    key = [combo['order_detail_id'], combo['transactions_type']]
    weight = (combo['price'] * combo['quantity']).clip(lower=0)
    all_priced = (weight > 0).groupby(key).transform('all')
    share = np.where(all_priced,
                        weight / weight.groupby(key).transform('sum'),
                        1 / weight.groupby(key).transform('size'))
    df_filtered.loc[is_combo, 'combo_share'] = share
    df_filtered.loc[is_combo, 'total_order_amount'] = combo['total_order_amount'] * share

def calculate_new_amount(group, amount_column):
    # 1 group = 1 order_detail dalam 1 transaksi -> tarif pajaknya sama (dari API)
    divisor = group['divisor'].iloc[0]
    if (group['type'] == 'comboset').all():
        # promo/diskon paket dibagi dengan bobot yang sama, supaya komponen murah tidak jadi minus
        return group[amount_column] * group['combo_share'] / divisor
    return group[amount_column] / len(group) / divisor

# Group per (order_detail_id, transactions_type): row sale & refund dihitung terpisah.
# Versi lama group per order_detail_id saja -> order ber-refund cuma kena 1/2 promo/diskon di row sale.
detail_key = ['order_detail_id', 'transactions_type']

# Diskon diambil dari total array `discounts[]` API (di-group per order_detail_id di cell 42 -> kolom discount_amount).
# `discount_order_amount` tidak lengkap: diskon per item "Custom ..." (diskon manual kasir) tidak masuk ke situ,
# termasuk saat 1 detail punya "Order Custom" + "Custom" sekaligus.
# Cek 1-14 Okt: total discounts[] per order = total_discount_amount header x 1.1 untuk 57,701/57,701 order.
df_filtered['discount_base'] = pd.to_numeric(df_filtered['discount_amount'], errors='coerce').fillna(0)

df_filtered['discount_amount_new'] = df_filtered.groupby(detail_key)['discount_base'].transform(
    lambda x: calculate_new_amount(df_filtered.loc[x.index], 'discount_base')
)

df_filtered['promotion_amount_new'] = df_filtered.groupby(detail_key)['promotion_amount'].transform(
    lambda x: calculate_new_amount(df_filtered.loc[x.index], 'promotion_amount')
)

# Menghitung subtotal, tax_amount, dan total_amount
df_filtered['subtotal'] = df_filtered['total_order_amount'] - df_filtered['discount_amount_new'] - df_filtered['promotion_amount_new']

# -- versi lama (tarif dari list produk):
# df_filtered['tax_amount'] = df_filtered.apply(
#     lambda row: row['subtotal'] * 0.11 if row['product_id'] in product_id_sjw_sjo_v2 else row['subtotal'] * 0.1, axis=1
# )
df_filtered['tax_amount'] = df_filtered['subtotal'] * df_filtered['tax_rate']

df_filtered['total_amount'] = df_filtered['subtotal'] + df_filtered['tax_amount']

# Drop temporary columns
df_filtered.drop(columns=['divisor', 'tax_rate', 'discount_base'], inplace=True)



In [ ]:
df_final_details = df_filtered[['order_id', 'transaction_date', 'order_reference', 'outlet_id',
       'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
       'closed_date', 'status', 'payment_status', 'transactions_type',
       'transactions_status', 'sales_type', 'transactions_payment_type_name',
       'customer_id', 'customer_first_name', 'customer_last_name',
       'customer_phone_number', 'customer_email', 'cashier_id',
       'order_detail_id', 'product_id', 'bundling_id', 'product_name',
       'product_generic_name', 'type', 'product_type', 'product_variant_id',
       'product_variant_name', 'product_modifier_id', 'product_modifier_name',
       'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
       'total_order_amount', 'discount_name', 'discount_percentage',
       'discount_amount_new', 'promotion_id','promotion_name', 
       'promotion_type', 'promotion_amount_new', 'subtotal',
       'taxable', 'tax_percentage', 'tax_amount', 'additional_charge','total_amount', 'modifier_product_sku']]

df_final_details.columns = ['order_id', 'transaction_date', 'order_reference', 'outlet_id',
       'outlet_code', 'outlet_name', 'channel', 'channel_detail', 'order_date',
       'closed_date', 'status', 'payment_status', 'transactions_type',
       'transactions_status', 'sales_type', 'transactions_payment_type_name',
       'customer_id', 'customer_first_name', 'customer_last_name',
       'customer_phone_number', 'customer_email', 'cashier_id',
       'order_detail_id', 'product_id', 'bundling_id', 'product_name',
       'product_generic_name', 'type', 'product_type', 'product_variant_id',
       'product_variant_name', 'product_modifier_id', 'product_modifier_name',
       'notes', 'sku', 'fulfillment_status', 'quantity', 'base_price',
       'total_order_amount', 'discount_name', 'discount_percentage',
       'discount_amount', 'promotion_id','promotion_name', 
       'promotion_type', 'promotion_amount', 'subtotal',
       'taxable', 'tax_percentage', 'tax_amount', 'additional_charge','total_amount', 'modifier_product_sku']

df_final_details['load_data_at'] = datetime.datetime.now()


In [ ]:
df_final_details = df_final_details[['order_id', 'order_detail_id', 'type', 'bundling_id', 'product_id', 'sku', 'product_name',
                 'product_generic_name', 'base_price', 'quantity', 'total_order_amount', 'promotion_name',
                 'promotion_amount', 'discount_amount', 'subtotal', 'product_variant_id', 'product_variant_name', 
                 'product_modifier_id', 'product_modifier_name', 'notes', 'taxable', 'tax_percentage', 'tax_amount',
                 'additional_charge', 'total_amount', 'load_data_at', 'product_type', 'discount_name', 'discount_percentage',
                 'promotion_id',  'promotion_type', 'transaction_date', 'order_reference', 'outlet_id','outlet_code',
                 'outlet_name', 'channel', 'channel_detail', 'order_date', 'closed_date', 'status', 'payment_status',
                 'transactions_type', 'transactions_status', 'sales_type', 'transactions_payment_type_name', 'customer_id', 
                 'customer_first_name', 'customer_last_name', 'customer_phone_number', 'customer_email', 'cashier_id', 
                 'fulfillment_status', 'modifier_product_sku']]

In [ ]:
# Mengupdate kolom 'sku' berdasarkan kondisi pada 'modifier_product_sku'

df_final_details.loc[df_final_details['modifier_product_sku'] != '', 'sku'] = df_final_details['modifier_product_sku']
df_final_details.drop(columns=['modifier_product_sku'], inplace=True)

In [ ]:
print(f"total {len(df_final_details)} details from {df_final_headers['order_id'].nunique()} orders")

In [ ]:
# Akurasi details vs headers.
# Order yang belum fulfilled (fulfillment_status = 'none') memang TIDAK punya details dari API (order_details kosong)
# -> dipisah, karena detailnya baru ditarik nanti oleh DAG etl_iseller_mitra_fulfillment / etl_iseller_mitra_backfilling setelah order fulfilled.
print('Transaction Date :', start_date , 's/d', end_date)
total_amount_headers = df_final_headers['total_amount'].sum()
total_amount_details = df_final_details['total_amount'].sum()

orders_with_details = set(df_final_details['order_id'])
hdr_with_details = df_final_headers[df_final_headers['order_id'].isin(orders_with_details)]
hdr_no_details = df_final_headers[~df_final_headers['order_id'].isin(orders_with_details)]
unfulfilled_no_details = hdr_no_details[hdr_no_details['fulfillment_status'] == 'none']
other_no_details = hdr_no_details[hdr_no_details['fulfillment_status'] != 'none']

print(f"Total Amount Headers : {total_amount_headers:,.2f}")
print(f"Total Amount Details : {total_amount_details:,.2f}")
print(f"Accuracy Total Amount: {(total_amount_details / total_amount_headers * 100).round(3)}% (semua order)")

print(f"\nOrder Fulfilled          : {hdr_with_details['order_id'].nunique():,} order")
print(f"  Total Amount Headers   : {hdr_with_details['total_amount'].sum():,.2f}")
print(f"  Total Amount Details   : {total_amount_details:,.2f}")
print(f"  Accuracy               : {(total_amount_details / hdr_with_details['total_amount'].sum() * 100).round(3)}%")

print(f"\nOrder unfulfilled tanpa details (wajar, nanti diurus backfill): {unfulfilled_no_details['order_id'].nunique():,} order, "
      f"total {unfulfilled_no_details['total_amount'].sum():,.2f}")
if len(other_no_details):
    print(f"PERLU DICEK: {other_no_details['order_id'].nunique():,} order SUDAH fulfilled / bukan 'none' tapi tidak punya details "
          f"(total {other_no_details['total_amount'].sum():,.2f}) -> lihat `other_no_details`")

In [ ]:
print(f"Total Order ID Duplicated : {len(df_final_headers[df_final_headers.duplicated('order_id')])}")

In [ ]:
headers_composite_key = [
    'order_id', 'transaction_id',
]
# df_final_headers grain-nya 1 row per TRANSAKSI, bukan per order. Order yang punya
# transaksi 'sale' + 'refund' memang SEHARUSNYA jadi 2 row (order_id sama, transaction_id beda).
# Dedup by ['order_id'] saja akan membuang row refund -> sisi header hitung 1x sementara
# sisi details hitung 2x (karena details_composite_key memuat 'transactions_type') -> timpang.
# Dengan transaction_id, kedua sisi punya grain yang sama dan rekonsiliasi seimbang.
# Duplikat asli (transaksi yang sama tertarik 2x) tetap terbuang karena transaction_id-nya sama.

print("=" * 80)
print("CLEANING - Remove duplicates by composite business key")
print("=" * 80)

# Cek ada berapa duplicate
headers_duplicate_rows = df_final_headers[df_final_headers.duplicated(subset=headers_composite_key, keep=False)]
print(f"Total duplicate rows: {len(headers_duplicate_rows)}")

## 5. Dump Data to Data Lake

In [ ]:
#dump data headers
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(hostsname =config_dl_dwh.hostname, 
                                                                             user=config_dl_dwh.username,
                                                                             pw=config_dl_dwh.password,
                                                                             db=config_dl_dwh.db_resource))

df_final_headers.to_sql('transactions_iseller_mitra', con = engine, if_exists = 'append', index=False)

In [ ]:
#dump data details
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(hostsname =config_dl_dwh.hostname, 
                                                                             user=config_dl_dwh.username,
                                                                             pw=config_dl_dwh.password,
                                                                             db=config_dl_dwh.db_resource))

df_final_details.to_sql('transactions_items_iseller_mitra', con = engine, if_exists = 'append', index=False)

## 6. Send Email Notification

In [ ]:
import smtplib, ssl

port = 587  # For starttls
smtp_server = "smtp.gmail.com"
sender_email = "saputra.christabel20@gmail.com"
receiver_email = ["athens.jiwagroup@gmail.com"]
import os, getpass
# Password SMTP Gmail: env var `GMAIL_SMTP_PASSWORD`, atau isi lewat prompt (jangan ditulis di notebook)
password = os.getenv("GMAIL_SMTP_PASSWORD") or getpass.getpass("GMAIL_SMTP_PASSWORD: ")

message = """\
Subject: [iSeller MITRA] - Insert Data Transactions & Item Transaction at {0} Success (New)

Hi ! This is reporting Data for You !

This message is sent from Jupyter Notebook.

Best Regards,
DatO ( Data autOmation )""".format(today_string)

context = ssl.create_default_context()
with smtplib.SMTP(smtp_server, port) as server:
    server.ehlo()  # Can be omitted
    server.starttls(context=context)
    server.ehlo()  # Can be omitted
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)

## 7. Validate JSON Missing 

In [ ]:
# #cek data missing dari hasil get data  

# data = get_data.copy()
# order_reference = '#964-6981'

# index = next((i for i, d in enumerate(data) if d and d[0].get('order_reference') == order_reference), None)

# if index is not None:
#     filtered_data = data[index]
#     print(filtered_data)
# else:
#     print(f"Data with order reference = {order_reference} not found.")